In [ ]:
import os
import pickle
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from langchain_groq import ChatGroq

FILE_PATH = os.path.dirname(os.path.abspath("."))
os.chdir(FILE_PATH)
load_dotenv()
from src.bm25 import BM25Search
from src.semantic import SemanticSearch
from src.hybrid import HybridSearch
from src.download_data import download_data
os.chdir(f"{FILE_PATH}/notebooks")

download_data()

CATEGORY = "Appliances"
PROCESSED_DATA_DIR = Path("../data/processed")
with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_product_documents.pkl", "rb") as f:
    documents = pickle.load(f)

with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_doc_ids.pkl", "rb") as f:
    doc_ids = pickle.load(f)

import duckdb

PROCESSED_DATA_DIR = Path("../data/processed")
product_data_file = "Appliances_products.parquet"

c2 = duckdb.connect()
products = c2.execute(f"SELECT * FROM read_parquet('{PROCESSED_DATA_DIR}/{product_data_file}')").df()

/Users/harrisonlee/miniforge3/envs/dsci575-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /Users/harrisonlee/Code/ubc-mds/Block 6/DSCI575/DSCI_575_project_hli76_wnsong/src
Review data for Appliances already downloaded
Meta data for Appliances already downloaded
Merged data for Appliances is ready
Products Data for Appliances is ready
Document id for Appliances is ready
Document id for Appliances is ready


In [2]:
print(products.keys())

Index(['parent_asin', 'product_title', 'main_category', 'store', 'price',
       'avg_rating', 'reviews', 'review_titles', 'helpful_votes'],
      dtype='str')


In [ ]:
TOP_K = 10



load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6352.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done


## Create RAG Pipeline using Semantic Retriever

```pseudocode
def pipeline(query):
    retriever = SemanticSearch(Documents)
    retrieved_products = retriever.search(query)
    context = build_docs(retrieved_products)
    prompt = build_prompt(query, context)
    return format_output(llm(prompt))
```

In [4]:
query = "Best container for my food that needs to be cold"

In [36]:
def build_context(results):
    context = ""
    for i, (index, score) in enumerate(results):
        product_asin = doc_ids[index]
        product_context = documents[index]
        # print(f"{i+1}. ({score:.3f}) {product.product_title.values[0]}")
        context += f"""
parent_asin: {product_asin}
{product_context}

"""
    return context

DEFAULT_SYSTEM_PROMPT = """
Instructions:
- You are a helpful Amazon shopping assistant.
- You must answer the question using ONLY the following context (real product reviews with helpful votes and the metadata for the products).
- Always cite the product ASIN when possible.
- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.
"""

def build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT):
    return f"""
{system_prompt}

---------

Context: 
{context}

---------

Question:
{query}

"""

def load_retrievers(documents):
    retrievers = {
        "bm25": BM25Search(documents), 
        "semantic": SemanticSearch(documents)
    }
    retrievers["hybrid"] = HybridSearch(
        bm25=retrievers["bm25"], 
        semantic=retrievers["semantic"], 
        alpha=0.5, 
        top_k_candidates=TOP_K + 100
    )
    return retrievers

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))

def RAG_pipeline(retriever, documents, query, llm, top_k=TOP_K):
    retriever = load_retrievers(documents)[retriever]
    if isinstance(retriever, HybridSearch):
        raw_results = retriever.search(query, top_k=top_k)
        results = [(idx, score) for idx, score, _details in raw_results]
    else:
        results = retriever.search(query, top_k=top_k)
    context = build_context(results)
    prompt = build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT)
    response = llm.invoke(prompt).content
    return response

print(RAG_pipeline("hybrid", documents, query, llm))

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4989.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
The best container for keeping food cold is the IceeNOW Freezer Pak (ASIN: B08JCZ4WNP) with a 4.6 average rating. A reviewer stated, "Great freezer container!!... I am not disappointed and I love the fact that it keeps my ice packs cold for a minimum of five/six hours." 

Alternatively, you may also consider the Friomex Dry Ice Packs (ASIN: B09Q8T47SM) with a 4.2 average rating. A reviewer mentioned, "Great for shipping!!: Worked great for what I needed and great value for the price. Worked as described." 

If you are looking for a portable option, the GiTenvy 15 Quarts 24 Cans Portable Car Cooler (ASIN: B0B3937GVL) may be suitable for you. A reviewer said, "It is electric!: The description on this cooler lacks one vital peace of information: it plugs into the cigarette lighter and runs like a mini fridge in your car." 

For storing eggs, you can consider the Whirlpool Egg Container (ASIN: B002ZNNQMO) with a 3.9 average rating or the Set of 3 Covered Egg Container